# Week 1, Day 4: A chat that remembers

**Lesson:** [week1/day4.ipynb](../../../week1/day4.ipynb), *Tokenizing with code, and the illusion of memory*. **About 35 minutes.**

**What you practice here**

- Tokens: getting a model's encoding with tiktoken, encoding text to token ids, and decoding each id back to its text.
- Calls are stateless, so the memory is a list you build: the conversation so far, with the model's replies as assistant messages, sent whole every time.
- What that costs: counting the tokens a conversation sends with each call.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week1/day4", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson shows two things. Text reaches a model as tokens, which tiktoken lets you see. And a model remembers
nothing between calls: it knows your name only if the earlier messages are sent again. Build a chat for a
wrestling-history assistant: a fan tells it their name and favorite match, then later asks for them back.

The functions that call a library or the model take it as their first parameter: `token_pieces` takes tiktoken,
`count_tokens` an encoding, and `chat` the client. They make the lesson's calls themselves. The checks hand them
stand-ins with the same methods, which record what you send, so nothing is downloaded and no model is called.

## What each call should give

```python
token_pieces(tiktoken, "Hi my name is Ed")
# a list of (token id, text) pairs, one per token, in order, such as [(..., 'Hi'), (..., ' my'), (..., ' name'), ...]

start = [{"role": "system", "content": "You are a friendly wrestling-history buff."}]
hello = "Hi! I'm Sam, and my favorite match is Hogan vs André at WrestleMania III."
reply = chat(client, start, hello)
# makes one request with the model gpt-4.1-mini and two messages: start's system message, then hello as a user
# message; returns the reply's text

conversation = remember(start, hello, reply)
# [start[0], {"role": "user", "content": hello}, {"role": "assistant", "content": reply}]

chat(client, conversation, "What's my favorite match?")
# this request sends all four messages, so the model can answer

count_tokens(encoding, conversation)
# the tokens of the three messages' contents, added up
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `token_pieces(tiktoken, text, model="gpt-4.1-mini")`

Splits a text into tokens with the model's encoding and pairs each token id with its text. `tiktoken` is the tiktoken
module; the checks pass a stand-in with the same functions. Get the encoding for `model` from tiktoken and encode
`text` into token ids. Then decode each id on its own, inside a list of one, to see its piece of text, as the
lesson's loop does. Return a list of `(token_id, token_text)` pairs, in order.

<details><summary>Hint</summary>

```python
encoding = tiktoken.encoding_for_model(model)
for token_id in encoding.encode(text):
    token_text = encoding.decode([token_id])
```

</details>

In [ ]:
def token_pieces(tiktoken, text, model="gpt-4.1-mini"):
    # Write token_pieces here, as described above.
    raise NotImplementedError("token_pieces is not written yet")

In [ ]:
check("token_pieces")

### 2 · `count_tokens(encoding, conversation)`

Counts the tokens a conversation sends: the tokens of every message's content, added up. `encoding` is what tiktoken
gives for a model, and `conversation` is a list of `{"role": ..., "content": ...}` dicts. Encode each message's
`"content"`, and add up how many token ids each gives. The API adds a few tokens for each message on top of these;
leave those out.

<details><summary>Hint</summary>

```python
len(encoding.encode(message["content"]))  # the tokens of one message
```

</details>

In [ ]:
def count_tokens(encoding, conversation):
    # Write count_tokens here, as described above.
    raise NotImplementedError("count_tokens is not written yet")

In [ ]:
check("count_tokens")

### 3 · `chat(client, conversation, message, model="gpt-4.1-mini")`

Sends the whole conversation plus your new message in one call and returns the reply's text. `conversation` is the
conversation so far: a list of messages that starts with a system message. Make one Chat Completions call with the
model `model` and the messages of `conversation`, followed by a user message holding `message`. Return the reply's
text. Leave `conversation` as it was: build a new list.

<details><summary>Hint</summary>

```python
messages = conversation + [{"role": "user", "content": message}]
response = client.chat.completions.create(model=model, messages=messages)
return response.choices[0].message.content
```

</details>

In [ ]:
def chat(client, conversation, message, model="gpt-4.1-mini"):
    # Write chat here, as described above.
    raise NotImplementedError("chat is not written yet")

In [ ]:
check("chat")

### 4 · `remember(conversation, message, reply)`

Returns the conversation with your message and the model's reply added at the end. This is how the chat gets its
memory. Return a new list: the messages of `conversation`, then a user message holding `message`, then an
`"assistant"` message holding `reply`. Leave `conversation` as it was.

<details><summary>Hint</summary>

```python
conversation + [{"role": "user", "content": message}, {"role": "assistant", "content": reply}]
```

</details>

In [ ]:
def remember(conversation, message, reply):
    # Write remember here, as described above.
    raise NotImplementedError("remember is not written yet")

In [ ]:
check("remember")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 1, Day 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Each turn with `gpt-4.1-mini` costs a fraction of a cent. The first `tiktoken.encoding_for_model` call downloads the
encoding once.

1. Load your key with `load_dotenv(override=True)` and make `client = OpenAI()`. Import `tiktoken` and make
   `encoding = tiktoken.encoding_for_model("gpt-4.1-mini")`.
2. Print `token_pieces(tiktoken, "Hulk Hogan body-slammed André the Giant at WrestleMania III")`, and see where the
   names split.
3. Make a starting conversation, `start`, holding one system message, and a first message, `hello`, that tells the
   model your name and favorite match. Keep `reply = chat(client, start, hello)` and print it. Then ask
   "What's my favorite match?" with the same `start`: the model cannot know, as each call is new.
4. Now make `conversation = remember(start, hello, reply)`, print `count_tokens(encoding, conversation)`, and ask
   again with `conversation`: this time the model knows. Every call sends (and pays for) the whole conversation, so
   the count grows with each turn.

<details><summary>Stuck? One way to write it</summary>

```python
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

print(token_pieces(tiktoken, "Hulk Hogan body-slammed André the Giant at WrestleMania III"))

start = [{"role": "system", "content": "You are a friendly wrestling-history buff."}]
hello = "Hi! I'm Sam, and my favorite match is Hogan vs André at WrestleMania III."
reply = chat(client, start, hello)
print(reply)
print(chat(client, start, "What's my favorite match?"))  # a new call with only start: it cannot know

conversation = remember(start, hello, reply)
print(count_tokens(encoding, conversation), "tokens in the conversation so far")
print(chat(client, conversation, "What's my favorite match?"))  # the whole conversation: now it knows
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.